# 31 CPU context, light CV, structured sequence, and light video

CPU。context baseline、clip 抽出、軽量 structured sequence、OpenCV の軽量動画 baseline をまとめます。

各 stage は `BASE_DIR/reports/preflight/compact_runs/` に JSONL log を追記します。既に期待 artifact が揃っている stage は skip します。再実行したい場合は `FORCE_RERUN_ALL=True` にしてください。

In [ ]:
from pathlib import Path
import importlib.util
import os
import sys


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


# v2 をデフォルトにします。v1 を再実行したい時だけここを書き換えてください。
os.environ.setdefault('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path(os.environ['BATTING_CODE_ROOT'])
    mydrive = Path('/content/drive/MyDrive')
    candidates = [
        fixed,
        Path('/content/drive/MyDrive/codex/batting_codex_handoff'),
        Path('/content/drive/MyDrive/batting_codex_handoff'),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            return candidate
    raise FileNotFoundError(
        'sport_pipeline repo が見つかりません。Drive mount と repo 配置を確認してください。\n'
        f'BATTING_CODE_ROOT={fixed}\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別配置の場合は compact notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/あなたの配置/batting_codex_handoff\n'
        f'MyDrive exists={mydrive.exists()}\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()
REPO_DIR = _resolve_repo_dir()
os.environ['BATTING_CODE_ROOT'] = str(REPO_DIR)
NOTEBOOK_DIR = REPO_DIR / 'notebooks'
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ['BASEBALL_VISION_RUN_PROFILE']
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME
src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(f'sport_pipeline を import できません: {src_dir}')

from sport_pipeline.compact_notebooks import CompactRunLogger
from sport_pipeline.pipeline.run_profile import artifact_id, load_run_profile, run_id

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
FULL_RUN_ID = run_id(RUN_PROFILE, 'full_run_id')
CONTEXT_RUN_ID = run_id(RUN_PROFILE, 'context_run_id')
SEQUENCE_RUN_ID = run_id(RUN_PROFILE, 'sequence_run_id')
SEQUENCE_TCN_RUN_ID = run_id(RUN_PROFILE, 'sequence_tcn_run_id')
VIDEO_LIGHTWEIGHT_RUN_ID = run_id(RUN_PROFILE, 'video_lightweight_run_id')
VIDEO_FROZEN_RUN_ID = run_id(RUN_PROFILE, 'video_frozen_run_id')
VIDEO_FINETUNE_RUN_ID = run_id(RUN_PROFILE, 'video_finetune_run_id')
PLAYER_SEASON_RUN_ID = run_id(RUN_PROFILE, 'player_season_run_id')
VLM_RUN_ID = run_id(RUN_PROFILE, 'vlm_run_id')
FUSION_RUN_ID = run_id(RUN_PROFILE, 'fusion_run_id')
METHOD_EVALUATION_REPORT_ID = run_id(RUN_PROFILE, 'method_evaluation_report_id')
VIDEO_ABLATION_REPORT_ID = run_id(RUN_PROFILE, 'video_ablation_report_id')
STRUCTURED_SEQUENCE_FEATURE_ID = artifact_id(RUN_PROFILE, 'structured_sequence_feature_id')
CLIP_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'clip_embedding_feature_id')
PLAYER_SEASON_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'player_season_embedding_feature_id')
SEQUENCE_DATASET_ID = artifact_id(RUN_PROFILE, 'sequence_dataset_id')
EVENT_WITH_PRIOR_DATASET_ID = artifact_id(RUN_PROFILE, 'event_with_prior_dataset_id')
VIDEO_LIGHTWEIGHT_FEATURE_ID = artifact_id(RUN_PROFILE, 'video_lightweight_feature_id', 'video_lightweight_features_mlb_2024_2026_v2')
VIDEO_EMBEDDING_FEATURE_ID = artifact_id(RUN_PROFILE, 'video_embedding_feature_id')
VLM_FEATURE_ID = artifact_id(RUN_PROFILE, 'vlm_feature_id')

print('RUN_PROFILE =', RUN_PROFILE_NAME)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('BASE_DIR =', BASE_DIR)
print('NOTEBOOK_DIR =', NOTEBOOK_DIR)
print('src_dir =', src_dir)


In [ ]:
COMPACT_RUN_NAME = '31_cpu_context_cv_sequence_light_video'
LOGGER = CompactRunLogger(BASE_DIR, COMPACT_RUN_NAME, run_profile_name=RUN_PROFILE_NAME)
FORCE_RERUN_ALL = False

RUN_CONTEXT_CATBOOST = True
RUN_FULL_CV_PREPROCESS = True
RUN_STRUCTURED_SEQUENCE_BASELINE = True
RUN_LIGHTWEIGHT_VIDEO_BASELINE = True

stages = [
    {
        'notebook': '05b_context_catboost_baseline.ipynb',
        'label': 'Context CatBoost event baseline',
        'enabled': RUN_CONTEXT_CATBOOST,
        'force': FORCE_RERUN_ALL,
        'expected_outputs': ['datasets/context_dataset_v1/manifest.parquet', f'predictions/{CONTEXT_RUN_ID}/predictions_v1.parquet', f'predictions/{CONTEXT_RUN_ID}/metrics_v1.json', f'reports/preflight/train_context_baseline_{CONTEXT_RUN_ID}_progress.json'],
        'progress_files': [f'reports/preflight/train_context_baseline_{CONTEXT_RUN_ID}_progress.json'],
    },
    {
        'notebook': '12_full_cv_preprocess.ipynb',
        'label': 'Full CV clip preprocessing',
        'enabled': RUN_FULL_CV_PREPROCESS,
        'force': FORCE_RERUN_ALL,
        'expected_outputs': [f'clips/{FULL_RUN_ID}/candidate_segments_v1.parquet', f'clips/{FULL_RUN_ID}/clips_v1.parquet', f'debug/{FULL_RUN_ID}/debug_overlays_v1.parquet', f'reports/preflight/full_cv_preprocess_{FULL_RUN_ID}.json', f'reports/preflight/full_cv_preprocess_{FULL_RUN_ID}_progress.json'],
        'progress_files': [f'reports/preflight/full_cv_preprocess_{FULL_RUN_ID}_progress.json'],
    },
    {
        'notebook': '13_full_sequence_baseline.ipynb',
        'label': 'Lightweight structured sequence baseline',
        'enabled': RUN_STRUCTURED_SEQUENCE_BASELINE,
        'force': FORCE_RERUN_ALL,
        'expected_outputs': [f'features/{STRUCTURED_SEQUENCE_FEATURE_ID}/manifest.parquet', f'features/{STRUCTURED_SEQUENCE_FEATURE_ID}/frames.parquet', f'features/{CLIP_EMBEDDING_FEATURE_ID}/manifest.parquet', f'features/{PLAYER_SEASON_EMBEDDING_FEATURE_ID}/manifest.parquet', f'datasets/{SEQUENCE_DATASET_ID}/manifest.parquet', f'datasets/{EVENT_WITH_PRIOR_DATASET_ID}/manifest.parquet', f'predictions/{SEQUENCE_RUN_ID}/predictions_v1.parquet', f'predictions/{SEQUENCE_RUN_ID}/metrics_v1.json', f'reports/preflight/full_sequence_baseline_{SEQUENCE_RUN_ID}.json'],
        'progress_files': [f'reports/preflight/full_sequence_baseline_{SEQUENCE_RUN_ID}.json'],
    },
    {
        'notebook': '14_full_video_baseline.ipynb',
        'label': 'Lightweight OpenCV video baseline',
        'enabled': RUN_LIGHTWEIGHT_VIDEO_BASELINE,
        'force': FORCE_RERUN_ALL,
        'expected_outputs': [f'features/{VIDEO_LIGHTWEIGHT_FEATURE_ID}/manifest.parquet', f'predictions/{VIDEO_LIGHTWEIGHT_RUN_ID}/predictions_v1.parquet', f'predictions/{VIDEO_LIGHTWEIGHT_RUN_ID}/metrics_v1.json', f'reports/preflight/full_video_baseline_{VIDEO_LIGHTWEIGHT_RUN_ID}.json'],
        'progress_files': [f'reports/preflight/full_video_baseline_{VIDEO_LIGHTWEIGHT_RUN_ID}.json'],
    },
]
LOGGER.run_stages(ipython=get_ipython(), notebook_dir=NOTEBOOK_DIR, stages=stages)
